In [44]:
import pandas as pd
import numpy as np

# Load season data
games = pd.read_excel("whl_2025.xlsx")

print(games.shape)
games.head()

(25827, 26)


,game_id,record_id,home_team,away_team,went_ot,home_off_line,home_def_pairing,away_off_line,away_def_pairing,home_goalie,...,home_goals,away_assists,away_shots,away_xg,away_max_xg,away_goals,home_penalties_committed,home_penalty_minutes,away_penalties_committed,away_penalty_minutes
0,game_1,record_1,thailand,pakistan,0,PP_kill_dwn,PP_kill_dwn,PP_up,PP_up,player_id_142,...,0,2,9,1.4645,0.2166,1,7,14,1,2
1,game_1,record_2,thailand,pakistan,0,second_off,second_def,second_off,second_def,player_id_142,...,0,2,1,0.0928,0.0928,1,0,0,0,0
2,game_1,record_3,thailand,pakistan,0,first_off,second_def,second_off,second_def,player_id_142,...,0,0,2,0.1880,0.0940,0,0,0,0,0
3,game_1,record_4,thailand,pakistan,0,second_off,first_def,second_off,first_def,player_id_142,...,0,0,1,0.0727,0.0727,0,0,0,0,0
4,game_1,record_5,thailand,pakistan,0,second_off,second_def,first_off,second_def,player_id_142,...,0,2,1,0.0769,0.0769,1,0,0,0,0


In [45]:
home_lines = pd.DataFrame({
    "Team": games["home_team"],
    "Line": games["home_off_line"],
    "xG": games["home_xg"]
})

away_lines = pd.DataFrame({
    "Team": games["away_team"],
    "Line": games["away_off_line"],
    "xG": games["away_xg"]
})

lines = pd.concat([home_lines, away_lines], ignore_index=True)

lines.head()

,Team,Line,xG
0,thailand,PP_kill_dwn,0.1754
1,thailand,second_off,0.0000
2,thailand,first_off,0.0000
3,thailand,second_off,0.1211
4,thailand,second_off,0.1207


In [46]:
line_perf = (
    lines.groupby(["Team","Line"])
    .agg(
        Avg_xG=("xG","mean"),
        Games=("xG","size")
    )
    .reset_index()
)

line_perf.head()

,Team,Line,Avg_xG,Games
0,brazil,PP_kill_dwn,0.184213,90
1,brazil,PP_up,1.121189,81
2,brazil,empty_net_line,0.100839,23
3,brazil,first_off,0.114445,708
4,brazil,second_off,0.115285,709


In [47]:
line1 = line_perf[line_perf["Line"] == 1]

secondary = (
    line_perf[line_perf["Line"] != 1]
    .groupby("Team")
    .agg(Secondary_xG=("Avg_xG","mean"))
    .reset_index()
)

In [ ]:
disparity = line1.merge(secondary, on="Team")

disparity["DisparityRatio"] = (
    disparity["Avg_xG"] / disparity["Secondary_xG"]
)

disparity = disparity.sort_values(
    "DisparityRatio",
    ascending=False
).reset_index(drop=True)

disparity.insert(0, "Rank", disparity.index + 1)

disparity[["Rank","Team","DisparityRatio"]].head(10)
print(line1.head())
print(line1.shape)

print(secondary.head())
print(secondary.shape)

,Rank,Team,DisparityRatio


In [49]:
disparity[["Rank","Team","DisparityRatio"]].head(10).to_csv(
    "phase1b_disparity.csv",
    index=False
)